# Model Training Tutorial: Fine-tuning Your Genomic AI Model 🏋️

Welcome to the most exciting part of our tutorial series! In the previous two tutorials, we prepared our data and initialized our model. Now it's time for **Model Training**.

> 💡 **Learning Objectives**: Understand supervised learning principles, master different trainer usage, complete end-to-end model fine-tuning

---

## What is Supervised Fine-tuning? 🎓

**Supervised fine-tuning** is the process of adapting a pre-trained model to a specific labeled dataset. The model "learns" through the following cycle:

```
1. 📊 Make predictions on sequences in the training set
2. 🎯 Compare predictions with true labels  
3. 📉 Calculate "error" or "loss"
4. 🔧 Adjust internal weights to reduce future errors
```

This cycle is repeated for all sequences in the training data across multiple "epochs".

### 🧠 Why is Fine-tuning So Effective?

| Training Stage | Knowledge Learned | Analogy |
|---------|------------|------|
| **Pre-training** | General patterns of genomic language | 📚 Learned to "read" genomes |
| **Fine-tuning** | Task-specific specialized knowledge | 🎯 Learned to "understand" translation efficiency |


## Training Components in OmniGenBench 🔧

To start training, we need to assemble several key components:

### 1. 🗂️ **Datasets and DataLoaders**
Wrap our training, validation, and test data into PyTorch `DataLoader`s that efficiently provide batched data to the model.

### 2. 📊 **Evaluation Metrics**
Define how we measure success. For classification tasks, we'll use the **F1 score** which balances precision and recall.

### 3. ⚙️ **Optimizer**
The algorithm that updates model weights. We'll use the popular **AdamW** optimizer.

### 4. 🎯 **Trainer**
OmniGenBench provides a powerful `Trainer` class that orchestrates the entire training process.

## 🚀 OmniGenBench Trainer Selection Guide

OmniGenBench provides three trainer backends to meet different needs:

| Trainer Type | Base Technology | Main Advantages | Best Use Cases |
|-----------|----------|----------|------------|
| **`Trainer`** (PyTorch Native) | Pure PyTorch | 🟢 Transparent and understandable<br>🟢 Full control | 🎯 Learning and understanding<br>🎯 Single GPU training |
| **`AccelerateTrainer`** | 🤗 Accelerate | 🟡 Seamless scaling<br>🟡 Distributed-friendly | 🎯 Multi-GPU/TPU<br>🎯 Large-scale training |
| **`HFTrainer`** | 🤗 Trainer | 🔴 Feature-rich<br>🔴 Complete ecosystem | 🎯 HF users<br>🎯 Standardized workflows |

**In this tutorial, we use `Trainer`** - it clearly demonstrates every step of the PyTorch training loop.

> 💭 **Selection Principles**: If you're a beginner or want deep understanding, choose `Trainer`; if you need scaling, choose `AccelerateTrainer`; if you're familiar with HF ecosystem, choose `HFTrainer`.

### 🛠️ Environment Setup and Data Preparation

First, let's set up our environment and rebuild the necessary components from previous tutorials.

In [ ]:
# Install required packages
!pip install torch transformers pandas autocuda multimolecule biopython scipy scikit-learn tqdm dill findfile requests omnigenbench -U

In [ ]:
import warnings
import torch
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from omnigenbench import (
    ClassificationMetric,
    OmniTokenizer,
    OmniModelForSequenceClassification,
    OmniDatasetForSequenceClassification,
    Trainer,
)

warnings.filterwarnings('ignore')
print("✅ Libraries imported successfully!")

### 📊 Configure Training Parameters

Let's define all training hyperparameters. This centralized configuration is a best practice for keeping experiments organized and reproducible.

In [ ]:
# Training Configuration
from dataclasses import dataclass

@dataclass
class TrainingConfig:
    MODEL_NAME = "yangheng/OmniGenome-52M"
    BATCH_SIZE = 8
    LEARNING_RATE = 1e-4
    NUM_EPOCHS = 3
    MAX_LENGTH = 512
    SAVE_DIR = "checkpoints/translation_efficiency"
    
    # Optional: Set random seeds for reproducibility  
    SEED = 42

config = TrainingConfig()
print("📋 Training configuration initialized!")
print(f"  🤖 Model: {config.MODEL_NAME}")
print(f"  ? Batch size: {config.BATCH_SIZE}")
print(f"  🎯 Learning rate: {config.LEARNING_RATE}")
print(f"  📊 Max sequence length: {config.MAX_LENGTH}")
print(f"  ? Save directory: {config.SAVE_DIR}")

### 🏗️ Assemble Training Components

Now, let's create all the objects needed for training. We'll reuse the dataset class from previous tutorials.

In [ ]:
# 1. Load tokenizer and model
print("🔄 Loading tokenizer and model...")
tokenizer = OmniTokenizer.from_pretrained(config.MODEL_NAME, trust_remote_code=True)
model = OmniModelForSequenceClassification(
    config.MODEL_NAME,
    tokenizer=tokenizer,
    label2id=config.LABEL2ID,
    trust_remote_code=True,
)
print(f"✅ Model loaded successfully! Parameter count: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

# 2. Create datasets (assuming you have data files ready)
# In actual usage, ensure your data files exist
print("📊 Preparing datasets...")
# train_set = OmniDatasetForSequenceClassification(
#     data_source="train.json",
#     tokenizer=tokenizer,
#     label2id=config.LABEL2ID,
#     max_length=config.MAX_LENGTH
# )
# ... similarly create valid_set and test_set

print("✅ DataLoaders prepared!")

### 📊 Define Evaluation Metrics and Optimizer

In [ ]:
# Set up training components
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
import os

# Create save directory
os.makedirs(config.SAVE_DIR, exist_ok=True)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Initialize optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=0.01)
scheduler = LinearLR(optimizer, start_factor=1.0, end_factor=0.1, total_iters=config.NUM_EPOCHS)

print(f"🎯 Training setup complete!")
print(f"  ? Device: {device}")
print(f"  📁 Save directory created: {config.SAVE_DIR}")
print(f"  ? Optimizer: AdamW")
print(f"  📈 Scheduler: LinearLR")

## 🚀 Start Training!

Now that all components are in place, we can initialize the `Trainer` and call the `.train()` method.

**The `Trainer` will automatically handle:**
- ✅ Moving data and model to the correct device (GPU or CPU)
- ✅ Iterating through training data for the specified number of epochs
- ✅ Calculating loss and updating model weights
- ✅ Evaluating the model on the validation set after each epoch
- ✅ Logging performance metrics
- ✅ Saving the best-performing model checkpoint on the validation set

In [ ]:
# Training loop
from tqdm import tqdm
import torch.nn.functional as F

def train_epoch(model, dataloader, optimizer, device):
    """Single training epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in tqdm(dataloader, desc="Training"):
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        predictions = torch.argmax(outputs.logits, dim=-1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(dataloader), correct / total

print("✅ Training function defined!")

### 📈 Visualizing Training Results

After training completes, we get a `metrics` dictionary containing performance on validation and test sets for each epoch. Visualizing these results is a great way to understand training progress.

In [ ]:
# Main training loop
print("🚀 Starting training...")

for epoch in range(config.NUM_EPOCHS):
    print(f"\n📊 Epoch {epoch + 1}/{config.NUM_EPOCHS}")
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, device)
    
    # Validate  
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for batch in tqdm(valid_loader, desc="Validating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            
            predictions = torch.argmax(outputs.logits, dim=-1)
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)
    
    val_loss /= len(valid_loader)
    val_acc = val_correct / val_total
    
    # Update scheduler
    scheduler.step()
    
    # Print results
    print(f"  📈 Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"  📊 Valid Loss: {val_loss:.4f}, Valid Acc: {val_acc:.4f}")
    print(f"  📚 Learning Rate: {scheduler.get_last_lr()[0]:.6f}")
    
    # Save checkpoint
    checkpoint_path = os.path.join(config.SAVE_DIR, f"epoch_{epoch+1}.pt")
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_acc': train_acc,
        'val_acc': val_acc,
    }, checkpoint_path)
    print(f"  ? Checkpoint saved: {checkpoint_path}")

print("\n🎉 Training completed!")

## 🎯 Summary and Next Steps

🎉 Congratulations! You have successfully completed the model training learning. Let's review what we've accomplished:

✅ **Mastered the core concepts of supervised fine-tuning**  
✅ **Understood the characteristics and selection principles of different trainers**  
✅ **Learned to assemble complete training workflows**  
✅ **Mastered training parameter configuration and best practices**  
✅ **Learned to visualize and analyze training results**  

**Your model is now "intelligent"!** 🧠✨

Through fine-tuning, we have transformed a general genomic foundation model into a translation efficiency prediction specialist. This trained model has been saved and is ready for actual predictions.

---

### 🚀 The Final Adventure...

In the final tutorial, we will explore:
- 🔮 How to use the trained model for inference
- 🌐 How to deploy the model to serve real applications
- 📊 How to evaluate and interpret model predictions
- 🚀 How to scale to production environments

**Ready to make your AI model serve the real world?**

> 🎊 **Milestone**: You are now a qualified genomic AI trainer!

👉 **Final Step**: Open `04_Inference_and_Deployment.ipynb` to complete your learning journey!